# Notebook 01: Exploratory Data Analysis (EDA)
## E-Commerce: Predicting Product Delivery Time for an Electronics Store

**Author:** hgabrali  
**Date:** 2026  

### Objectives
1. Load and inspect the merged e-commerce dataset.
2. Understand the distribution of the target variable (on-time vs. delayed).
3. Analyze numeric and categorical feature distributions.
4. Identify correlations between features and the target.
5. Uncover seasonal or temporal patterns in delivery performance.
6. Generate actionable insights to guide feature engineering.

In [ ]:
# Standard library
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Project utilities
import sys
sys.path.append('../')
from src.data_preprocessing import load_data, engineer_features, clean_data
from src.visualization import (
    plot_target_distribution,
    plot_correlation_heatmap,
)

# Notebook settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
sns.set_style('whitegrid')
sns.set_palette('husl')
%matplotlib inline

print('All imports successful.')

## 1. Load Data

In [ ]:
DATA_PATH = '../data/raw/ecommerce_dataset.csv'

# Load raw data
df_raw = load_data(DATA_PATH)
print(f'Dataset shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head(3)

## 2. Basic Statistics

In [ ]:
print('=== Dataset Info ===')
df_raw.info()
print('\n=== Missing Values ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0])

In [ ]:
print('=== Descriptive Statistics (Numeric Features) ===')
df_raw.describe().T

## 3. Feature Engineering & Target Creation

In [ ]:
df = engineer_features(df_raw)
df = clean_data(df)
print(f'After engineering and cleaning: {df.shape}')
print(f'Target distribution:')
print(df['delivery_time_class'].value_counts(normalize=True).round(3))

## 4. Target Variable Distribution

In [ ]:
fig = plot_target_distribution(df['delivery_time_class'], save=False)
plt.show()

## 5. Delivery Days Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual delivery days
axes[0].hist(df['actual_delivery_days'].dropna(), bins=50,
             color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Actual Delivery Days', fontsize=13)
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Frequency')

# Delivery delay
axes[1].hist(df['delivery_delay_days'].dropna(), bins=50,
             color='#F44336', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='black', linestyle='--', label='On-Time Boundary')
axes[1].set_title('Distribution of Delivery Delay (actual - estimated)', fontsize=13)
axes[1].set_xlabel('Delay (Days)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Categorical Feature Analysis

In [ ]:
cat_cols = ['product_category_name', 'payment_type', 'customer_state']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, cat_cols):
    top_vals = df[col].value_counts().head(10)
    ax.barh(top_vals.index[::-1], top_vals.values[::-1], color='#5C6BC0')
    ax.set_title(f'Top 10 {col}', fontsize=11)
    ax.set_xlabel('Count')

plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

In [ ]:
numeric_cols = [
    'product_weight_g', 'freight_value', 'price', 'payment_value',
    'product_volume_cm3', 'price_per_gram', 'actual_delivery_days',
    'estimated_delivery_days', 'delivery_delay_days', 'review_score',
    'delivery_time_class'
]
existing = [c for c in numeric_cols if c in df.columns]
fig = plot_correlation_heatmap(df[existing], save=False)
plt.show()

## 8. Temporal Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Delay rate by purchase hour
hourly = df.groupby('purchase_hour')['delivery_time_class'].mean()
axes[0].bar(hourly.index, hourly.values, color='#FF9800', edgecolor='white')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
axes[0].set_title('Late Delivery Rate by Purchase Hour', fontsize=12)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Delay Rate')

# Delay rate by weekday
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily = df.groupby('purchase_dayofweek')['delivery_time_class'].mean()
axes[1].bar(daily.index, daily.values, color='#9C27B0', edgecolor='white')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_labels)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
axes[1].set_title('Late Delivery Rate by Weekday', fontsize=12)
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Delay Rate')

plt.tight_layout()
plt.show()

## 9. Key Insights

Summarize the most important findings from the EDA here, e.g.:
- Class imbalance: ~X% on-time, ~Y% delayed
- Strongest correlations with the target
- Seasonal patterns in delivery delays
- Feature engineering opportunities identified